
####Q1 (easy): Days between a customer's consecutive orders

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

fact = spark.table("retail_sales_dev.gold.fact_sales")

In [0]:
%sql
WITH invoices AS (
  SELECT CustomerSK,
         InvoiceNumber,
         MIN(InvoiceDate)  AS order_ts,
         SUM(TotalAmount)  AS order_value
  FROM retail_sales_dev.gold.fact_sales
  WHERE CustomerSK <> -1                       -- replace with however your unknown customer is keyed
    AND InvoiceNumber NOT LIKE 'C%'            -- exclude cancellations (or use your is_cancelled flag if it's in Gold)
  GROUP BY CustomerSK, InvoiceNumber
),
with_prev AS (
  SELECT *,
         LAG(order_ts) OVER (PARTITION BY CustomerSK
                             ORDER BY order_ts, InvoiceNumber) AS prev_order_ts
  FROM invoices
)
SELECT CustomerSK,
       InvoiceNumber,
       order_ts,
       prev_order_ts,
       DATEDIFF(order_ts, prev_order_ts) AS days_since_prev   -- NULL for a customer's first order
FROM with_prev
ORDER BY CustomerSK, order_ts;

In [0]:
invoices = (fact
    .groupBy("CustomerSK", "InvoiceNumber")
    .agg(F.min("InvoiceDate").alias("order_ts"),
         F.sum("TotalAmount").alias("order_value")))

w = Window.partitionBy("CustomerSK").orderBy("order_ts")

(invoices
  .withColumn("prev_order_ts", F.lag("order_ts").over(w))
  .withColumn("days_since_prev", F.datediff("order_ts", "prev_order_ts"))
  .show())

####Q2: Month-over-month revenue growth by country

In [0]:
%sql
WITH CTE AS(
SELECT COUNTRY,trunc(InvoiceDate, 'MM') as Month,SUM(ROUND(PRICE*Quantity)) AS Revenue FROM retail_sales_dev.silver.sales_cleaned
WHERE not is_cancelled
GROUP BY COUNTRY, trunc(InvoiceDate, 'MM')
),
CTE2 AS(SELECT *,lag(Revenue) over (partition by Country order by Month) as prev_revenue
FROM CTE)
SELECT *, (Revenue - prev_revenue)/prev_revenue* 100 AS growth FROM CTE2


In [0]:
silver = spark.table("retail_sales_dev.silver.sales_cleaned")

monthly = (silver.filter("is_cancelled = false")
    .withColumn("month", F.date_trunc("month", "InvoiceDate"))
    .groupBy("Country", "month")
    .agg(F.sum(F.col("Price") * F.col("Quantity")).alias("revenue")))

w = Window.partitionBy("Country").orderBy("month")

(monthly
  .withColumn("prev_rev", F.lag("revenue").over(w))
  .withColumn("mom_pct", F.round((F.col("revenue") - F.col("prev_rev")) / F.col("prev_rev") * 100, 2))
  .show())